<div align="center">
<h1>🔐 KRİPTOQRAFİYA KURSU</h1>
<h2>Məşğələ 12b</h2>
<h2>CTR, GCM və Müasir Rejimlər</h2>
<h3 style="color: #8B4513;">AEAD rejimləri - məxfilik və tamlıq bir yerdə</h3>
<br>
<h3>Məşğələ vaxtı: 2 saat</h3>
<br>
<p><em>Hazırlanma tarixi: 2024</em></p>
</div>

## 📑 Mündəricat

1. [Məşğələnin məqsədləri](#məşğələnin-məqsədləri)
2. [Lazım olan kitabxanalar](#lazım-olan-kitabxanalar)
3. [Hazırlıq (15 dəq)](#hazırlıq-15-dəq)
4. [Xatırlatma: İş rejimləri və onların təkamülü (10 dəq)](#xatırlatma-iş-rejimləri-və-onların-təkamülü-10-dəq)
5. [CTR (Counter) Rejimi (25 dəq)](#ctr-counter-rejimi-25-dəq)
6. [AEAD (Authenticated Encryption with Associated Data) (15 dəq)](#aead-authenticated-encryption-with-associated-data-15-dəq)
7. [GCM (Galois/Counter Mode) (30 dəq)](#gcm-galois-counter-mode-30-dəq)
8. [GCM-SIV: Nonce Təkrarına Davamlı AEAD (15 dəq)](#gcm-siv-nonce-təkrarına-davamlı-aead-15-dəq)
9. [CCM (Counter with CBC-MAC) Rejimi (15 dəq)](#ccm-counter-with-cbc-mac-rejimi-15-dəq)
10. [XTS Rejimi: Disk Şifrələməsi (15 dəq)](#xts-rejimi-disk-şifrələməsi-15-dəq)
11. [Müasir Protokollarda AEAD Rejimləri (15 dəq)](#müasir-protokollarda-aead-rejimləri-15-dəq)
12. [AEAD Rejimlərinin Təhlükəsizlik Müqayisəsi (5 dəq)](#aead-rejimlərinin-təhlükəsizlik-müqayisəsi-5-dəq)
13. [Praktik Tövsiyələr (10 dəq)](#praktik-tövsiyələr-10-dəq)
14. [İnteqrasiya edilmiş tətbiq (15 dəq)](#inteqrasiya-edilmiş-tətbiq-15-dəq)
15. [Ev tapşırığı](#ev-tapşırığı)
16. [Yekun və müzakirə sualları](#yekun-və-müzakirə-sualları)
17. [Əlavə resurslar](#əlavə-resurslar)

## 🎯 Məşğələnin məqsədləri

Bu məşğələni bitirdikdən sonra siz:

✅ CTR rejiminin iş prinsipini, üstünlüklərini və təhlükəsizlik xüsusiyyətlərini izah edə biləcəksiniz  
✅ AEAD (Authenticated Encryption with Associated Data) konsepsiyasını başa düşəcəksiniz  
✅ GCM rejiminin quruluşunu, GHASH funksiyasını implementasiya edə biləcəksiniz  
✅ GCM-SIV, CCM, XTS kimi müasir rejimlərin xüsusiyyətlərini biləcəksiniz  
✅ Müasir protokollarda (TLS 1.3, IPsec, WireGuard) AEAD rejimlərinin necə istifadə edildiyini izah edə biləcəksiniz  
✅ AEAD rejimlərinin düzgün implementasiyası üçün praktik tövsiyələri biləcəksiniz

## 📚 Lazım olan kitabxanalar

| Kitabxana | Quraşdırma | İstifadə sahəsi |
|-----------|------------|-----------------|
| PyCryptodome | `!pip install pycryptodome` | AES, GCM, CCM dəstəyi |
| cryptography | `!pip install cryptography` | AEAD rejimləri, ChaCha20-Poly1305 |
| Python standart kitabxanası | `os`, `struct` (daxili) | Təsadüfi ədədlər, byte çevirmələri |

> 💡 **Qeyd:** PyCryptodome və ya cryptography kitabxanalarından biri mütləq quraşdırılmalıdır.

In [ ]:
# Lazımi kitabxanaların quraşdırılması

!pip install pycryptodome cryptography --quiet

print("✅ Kitabxanalar quraşdırıldı")

## 🔧 Hazırlıq (15 dəq)

### 3.1 Python mühitinin yoxlanılması

Aşağıdakı kodu işə salaraq lazımi modulların yükləndiyini yoxlayın:

In [ ]:
import sys
import os
import struct
import math
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from collections import Counter

# PyCryptodome kitabxanası
try:
    from Crypto.Cipher import AES
    from Crypto.Random import get_random_bytes
    from Crypto.Util.Padding import pad, unpad
    from Crypto.Protocol.KDF import PBKDF2
    print("✅ PyCryptodome yüklü - AES, GCM istifadə edilə bilər")
    CRYPTO_AVAILABLE = True
except ImportError:
    print("❌ PyCryptodome yoxdur!")
    CRYPTO_AVAILABLE = False

# cryptography kitabxanası
try:
    from cryptography.hazmat.primitives.ciphers.aead import (
        AESGCM, ChaCha20Poly1305, AESCCM
    )
    from cryptography.hazmat.primitives import hashes
    from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
    print("✅ cryptography yüklü - AEAD rejimləri istifadə edilə bilər")
    CRYPTOGRAPHY_AVAILABLE = True
except ImportError:
    print("❌ cryptography yoxdur!")
    CRYPTOGRAPHY_AVAILABLE = False

print(f"\n🐍 Python versiyası: {sys.version}")

### 3.2 İşçi qovluğun yaradılması

In [ ]:
from pathlib import Path
import os

# İşçi qovluğu yaradaq (təkrar icrada iç-içə qovluq yaratmasın)
if "NOTEBOOK_ROOT" not in globals():
    NOTEBOOK_ROOT = Path.cwd().resolve()

workspace = NOTEBOOK_ROOT / "crypto_workshop" / "lecture12"
workspace.mkdir(parents=True, exist_ok=True)
os.chdir(workspace)

print(f"📁 İşçi qovluq: {Path.cwd()}")
print(f"📂 Qovluğun məzmunu: {os.listdir('.')}")

## 📖 Xatırlatma: İş rejimləri və onların təkamülü (10 dəq)

<div style="background-color: #fff3e0; padding: 15px; border-radius: 10px; border-left: 5px solid #ff9800;">
<h4>📜 İş rejimlərinin təkamülü</h4>
<ul>
    <li><b>1970-1980</b>: ECB, CBC, CFB, OFB (yalnız məxfilik)</li>
    <li><b>1990-2000</b>: CTR (paralel şifrləmə), CCM (AEAD)</li>
    <li><b>2004</b>: GCM (yüksək sürətli AEAD)</li>
    <li><b>2015</b>: GCM-SIV (nonce təkrarına davamlı)</li>
    <li><b>2018</b>: ChaCha20-Poly1305 (TLS 1.3-də standart)</li>
</ul>
</div>

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4; margin-top: 10px;">
<h4>🔑 Niyə yeni rejimlərə ehtiyac var?</h4>
<ol>
    <li><b>Paralellik</b>: Müasir prosessorlar çoxnüvəlidir</li>
    <li><b>AEAD</b>: Məxfilik + tamlıq eyni anda tələb olunur</li>
    <li><b>Yüksək sürət</b>: Gigabit şəbəkələr üçün</li>
    <li><b>Təhlükəsizlik</b>: Nonce təkrarına davamlılıq</li>
</ol>
</div>

## 🔢 CTR (Counter) Rejimi (25 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔢 CTR rejiminin iş prinsipi</h4>
<p>CTR rejimində hər bir məlumat bloku üçün sayğac dəyəri şifrlənir və alınan nəticə açıq mətnlə XOR edilir:</p>

<p style="text-align: center; font-size: 1.2em;">$$C_i = P_i \oplus E_K(IV \parallel i)$$</p>

<p><b>Üstünlüklər:</b></p>
<ul>
    <li>✅ Paralel şifrləmə mümkün</li>
    <li>✅ Təsadüfi giriş imkanı</li>
    <li>✅ Padding tələb etmir</li>
    <li>✅ Çox sürətli</li>
</ul>

<p><b>Zəifliklər:</b></p>
<ul>
    <li>❌ Nonce təkrarı təhlükəlidir (OTP kimi)</li>
    <li>❌ Tamlıq təmin etmir</li>
</ul>
</div>

In [ ]:
import os


def toy_block_encrypt(block, key):
    """
    Təhsil məqsədli oyuncaq blok "şifrəsi".
    Bu, AES deyil və real təhlükəsizlik üçün istifadə OLUNMAMALIDIR.
    """
    result = bytearray()
    for i, b in enumerate(block):
        result.append(b ^ key[i % len(key)])
    return bytes(result)


def aes_block_encrypt(block, key):
    """
    16-baytlıq bloku AES ilə şifrləyir.

    Qeyd: Aşağıdakı AES.MODE_ECB yalnız CTR rejimində sayğac blokunu
    AES blok primitivindən keçirmək üçündür. ECB rejimini mesajları birbaşa
    şifrləmək üçün istifadə etmək təhlükəsiz deyil: eyni açıq mətn blokları
    eyni şifrəli bloklara çevrilir və struktur sızdırır.
    """
    if len(block) != 16:
        raise ValueError("Blok uzunluğu 16 bayt olmalıdır")
    if len(key) not in {16, 24, 32}:
        raise ValueError("AES açarı 16, 24 və ya 32 bayt olmalıdır")

    if not CRYPTO_AVAILABLE:
        # PyCryptodome yoxdursa yalnız tədris məqsədli fallback
        return toy_block_encrypt(block, key)

    cipher = AES.new(key, AES.MODE_ECB)
    return cipher.encrypt(block)


def ctr_encrypt(plaintext, key, iv, block_cipher):
    """
    CTR rejimində şifrləmə.

    Parametrlər:
        plaintext: şifrlənəcək mətn (bytes)
        key: açar
        iv: 16 bayt = 12 bayt nonce + 4 bayt başlanğıc sayğacı
        block_cipher: 16-bayt blok şifrləmə funksiyası

    Qaytarır:
        ciphertext: şifrəli mətn
    """
    if len(iv) != 16:
        raise ValueError("CTR nümunəsində IV tam 16 bayt olmalıdır (12 bayt nonce + 4 bayt sayğac)")

    block_size = 16
    nonce = iv[:12]
    counter = int.from_bytes(iv[12:], "big")
    ciphertext = bytearray()

    for i in range(0, len(plaintext), block_size):
        if counter >= 2**32:
            raise OverflowError("32-bit sayğac doldu")

        counter_block = nonce + counter.to_bytes(4, "big")
        keystream = block_cipher(counter_block, key)

        block = plaintext[i:i + block_size]
        ciphertext.extend(a ^ b for a, b in zip(block, keystream))
        counter += 1

    return bytes(ciphertext)


def ctr_decrypt(ciphertext, key, iv, block_cipher):
    """
    CTR rejimində deşifrləmə (eyni funksiya).
    """
    return ctr_encrypt(ciphertext, key, iv, block_cipher)


# Test
print("=" * 70)
print("🔢 CTR REJİMİ TESTİ")
print("=" * 70)

key = get_random_bytes(16) if CRYPTO_AVAILABLE else os.urandom(16)
nonce = get_random_bytes(12) if CRYPTO_AVAILABLE else os.urandom(12)
initial_counter = 0
iv = nonce + initial_counter.to_bytes(4, "big")

plaintext = "Salam, dünya! Bu CTR test mesajıdır. Azərbaycan dili: ə, ü, ö, ç, ş, ğ".encode("utf-8")
block_cipher = aes_block_encrypt

print(f"📨 Açıq mətn (UTF-8): {plaintext.decode('utf-8')}")

cipher = ctr_encrypt(plaintext, key, iv, block_cipher)
decrypted = ctr_decrypt(cipher, key, iv, block_cipher)

print(f"🔒 Şifrəli (hex): {cipher.hex()}")
print(f"🔓 Deşifrə (UTF-8): {decrypted.decode('utf-8')}")
print(f"\n✅ Uğurlu? {plaintext == decrypted}")

### 5.1 CTR rejiminin paralel şifrləməsi

In [ ]:
def ctr_encrypt_parallel(plaintext, key, iv, block_cipher, num_threads=4):
    """
    CTR rejimində paralel şifrləmə.

    Parametrlər:
        plaintext: şifrlənəcək mətn (bytes)
        key: açar
        iv: 16 bayt = 12 bayt nonce + 4 bayt başlanğıc sayğacı
        block_cipher: 16-bayt blok şifrləmə funksiyası
        num_threads: thread sayı

    Qaytarır:
        ciphertext: şifrəli mətn
    """
    if len(iv) != 16:
        raise ValueError("IV 16 bayt olmalıdır")

    block_size = 16
    nonce = iv[:12]
    start_counter = int.from_bytes(iv[12:], "big")

    num_blocks = (len(plaintext) + block_size - 1) // block_size
    if num_blocks == 0:
        return b""

    blocks_per_thread = (num_blocks + num_threads - 1) // num_threads
    results = [b""] * num_blocks

    def encrypt_chunk(thread_id):
        start_block = thread_id * blocks_per_thread
        end_block = min(start_block + blocks_per_thread, num_blocks)

        for block_idx in range(start_block, end_block):
            counter = start_counter + block_idx
            if counter >= 2**32:
                raise OverflowError("32-bit sayğac doldu")

            counter_block = nonce + counter.to_bytes(4, "big")
            keystream = block_cipher(counter_block, key)

            start_byte = block_idx * block_size
            end_byte = min(start_byte + block_size, len(plaintext))
            block = plaintext[start_byte:end_byte]
            results[block_idx] = bytes(a ^ b for a, b in zip(block, keystream))

    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        futures = [executor.submit(encrypt_chunk, i) for i in range(num_threads)]
        for future in futures:
            future.result()

    return b"".join(results)

print("\n" + "-" * 70)
print("⚡ CTR PARALEL ŞİFRLƏMƏ")
print("-" * 70)
print("CTR rejimi paralel şifrləməyə imkan verir!")

sample = b"Parallel CTR test message " * 8
seq_result = ctr_encrypt(sample, key, iv, aes_block_encrypt)
par_result = ctr_encrypt_parallel(sample, key, iv, aes_block_encrypt, num_threads=4)

print(f"✅ Ardıcıl və paralel nəticələr eynidir? {seq_result == par_result}")

### 5.2 Nonce təkrarı problemi

<div style="background-color: #ffebee; padding: 15px; border-radius: 10px; border-left: 5px solid #f44336;">
<h4>⚠️ CTR-də nonce təkrarı</h4>
<p>Əgər eyni $(K, IV)$ cütü iki fərqli mesaj üçün istifadə olunarsa:</p>
<p style="text-align: center; font-size: 1.2em;">$$C_1 \oplus C_2 = P_1 \oplus P_2$$</p>
<p>Bu, hücumçuya iki mesajın XOR-unu verir.</p>
</div>

In [ ]:
def nonce_reuse_demo():
    """
    Eyni nonce-un təkrar istifadəsi problemini göstərir
    """
    import secrets

    print("\n" + "-" * 70)
    print("⚠️ NONCE TƏKRARI PROBLEMİ")
    print("-" * 70)

    key = secrets.token_bytes(16)
    nonce = secrets.token_bytes(12)  # Təsadüfi seçilir, amma aşağıda qəsdən təkrar istifadə olunur
    counter = 0
    iv = nonce + counter.to_bytes(4, "big")

    message1 = b"Gizli mesaj A - 12345"
    message2 = b"Gizli mesaj B - 67890"

    c1 = ctr_encrypt(message1, key, iv, aes_block_encrypt)
    c2 = ctr_encrypt(message2, key, iv, aes_block_encrypt)

    print(f"📨 Mesaj 1: {message1}")
    print(f"📨 Mesaj 2: {message2}")
    print(f"\n🔒 c1: {c1.hex()}")
    print(f"🔒 c2: {c2.hex()}")

    xor_c1_c2 = bytes(a ^ b for a, b in zip(c1, c2))
    xor_m1_m2 = bytes(a ^ b for a, b in zip(message1, message2))

    print(f"\n🔢 c1 ⊕ c2: {xor_c1_c2.hex()}")
    print(f"🔢 m1 ⊕ m2: {xor_m1_m2.hex()}")
    print(f"\n✅ Bərabərdir? {xor_c1_c2 == xor_m1_m2}")
    print("\n🚨 TƏHLÜKƏ: Nonce təkrarı bütün təhlükəsizliyi çökdürür!")

nonce_reuse_demo()


### ✍️ Çalışma 1: CTR rejimi (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Sayğac dəyərləri:** $IV = \text{0x0123456789ABCDEF0011223300000000}$ (12 bayt nonce + 4 bayt sayğac) üçün ilk 5 blokun sayğac dəyərlərini yazın.

2. **CTR vs CBC:** CTR rejimini CBC ilə müqayisə edin:
   * Hansı paralel şifrləməyə imkan verir?
   * Hansı təsadüfi girişə imkan verir?
   * Hansı doldurma (padding) tələb etmir?

3. **Nonce təkrarı:** Eyni $IV$ iki fərqli mesaj üçün istifadə edildikdə nə baş verir? Nümunə ilə göstərin.

4. **Paralel şifrləmə:** CTR rejimində paralel şifrləməni threading ilə implementasiya edin.


In [ ]:
# Çalışma 1 - Cavablar

print("📝 ÇALIŞMA 1 CAVABLARI")
print("=" * 80)

# 1. Sayğac dəyərləri
print("\n1. SAYĞAC DƏYƏRLƏRİ:")
nonce_hex = "00112233445566778899aabb"  # 12 bayt nonce = 24 hex simvol
counter_start = 0
print(f"   Nonce: {nonce_hex}")
print("   Sayğac blokları:")
for i in range(5):
    print(f"     Blok {i}: {nonce_hex}{counter_start + i:08x}")

# 2. CTR vs CBC
print("\n2. CTR vs CBC:")
print("""
   | Xüsusiyyət        | CTR          | CBC          |
   |-------------------|--------------|--------------|
   | Paralel şifrləmə  | ✅ Bəli      | ❌ Xeyr      |
   | Təsadüfi giriş    | ✅ Bəli      | ❌ Xeyr      |
   | Padding tələbi    | ❌ Xeyr      | ✅ Bəli      |
   | Xəta yayılması    | ❌ Yox       | 🔄 2 blok    |
""")

# 3. Nonce təkrarı
print("\n3. NONCE TƏKRARI:")
print("   Yuxarıdakı nonce_reuse_demo() funksiyası bunu göstərir.")
print("   Eyni açar və eyni nonce istifadə edilərsə, c1 ⊕ c2 = m1 ⊕ m2 olur.")

# 4. Paralel şifrləmə
print("\n4. PARALEL ŞİFRLƏMƏ:")
print("   ctr_encrypt_parallel() funksiyası ardıcıl CTR nəticəsi ilə eyni çıxışı verir.")

## 🔐 AEAD (Authenticated Encryption with Associated Data) (15 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔑 AEAD konsepsiyası</h4>
<p>AEAD rejimləri üç funksiyanı bir yerə birləşdirir:</p>
<ul>
    <li><b>Məxfilik:</b> Məlumat şifrlənir</li>
    <li><b>Tamlıq:</b> Məlumatın dəyişdirilmədiyi təsdiqlənir</li>
    <li><b>Autentifikasiya:</b> Məlumatın həqiqi mənbədən gəldiyi təsdiqlənir</li>
</ul>

<p>Əlavə olaraq, <b>əlaqəli verilənlər (associated data)</b> anlayışı təqdim edilir — şifrlənməyən, lakin tamlığı qorunmalı olan məlumat.</p>
</div>

<div style="background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-left: 5px solid #4caf50; margin-top: 10px;">
<h4>📌 Əlaqəli verilənlər nümunələri</h4>
<ul>
    <li>📦 Şəbəkə paketinin başlığı (IP ünvanı, port nömrəsi)</li>
    <li>⏱️ Protokol parametrləri (nonce, timestamp)</li>
    <li>📁 Metadata (fayl adı, fayl ölçüsü)</li>
</ul>
</div>

In [ ]:
def derive_demo_keys(master_key):
    """
    Tədris məqsədi ilə master açardan iki ayrıca açar törədir:
    biri şifrləmə, biri MAC üçün.
    """
    import hashlib

    enc_key = hashlib.sha256(master_key + b":enc").digest()[:16]
    mac_key = hashlib.sha256(master_key + b":mac").digest()
    return enc_key, mac_key

def simple_auth_tag(mac_key, nonce, aad, ciphertext):
    """
    Tədris məqsədi ilə HMAC-SHA256 əsaslı teq.
    Qeyd: Bu GCM/Poly1305 deyil; generic Encrypt-then-MAC nümunəsidir.
    AAD və ciphertext sərhədləri uzunluq kodlaması ilə autentifikasiya olunur.
    """
    import hmac
    import hashlib

    mac = hmac.new(mac_key, digestmod=hashlib.sha256)
    mac.update(b"EtM-demo-v1")
    mac.update(nonce)
    mac.update(len(aad).to_bytes(8, "big"))
    mac.update(aad)
    mac.update(len(ciphertext).to_bytes(8, "big"))
    mac.update(ciphertext)
    return mac.digest()[:16]

def aead_encrypt(master_key, nonce, aad, plaintext):
    """
    Generic Encrypt-then-MAC sxemi.
    Bu, xüsusi AEAD rejimi (məs., GCM/CCM) deyil, amma düzgün kompozisiyanı göstərir.

    Returns: (ciphertext, tag)
    """
    if len(nonce) != 12:
        raise ValueError("Nonce 12 bayt olmalıdır")

    enc_key, mac_key = derive_demo_keys(master_key)
    iv = nonce + (0).to_bytes(4, "big")
    ciphertext = ctr_encrypt(plaintext, enc_key, iv, aes_block_encrypt)
    tag = simple_auth_tag(mac_key, nonce, aad, ciphertext)
    return ciphertext, tag

def aead_decrypt(master_key, nonce, aad, ciphertext, tag):
    """
    Tag-i yoxladıqdan sonra deşifrləyir.
    """
    import hmac

    enc_key, mac_key = derive_demo_keys(master_key)
    expected_tag = simple_auth_tag(mac_key, nonce, aad, ciphertext)

    if not hmac.compare_digest(expected_tag, tag):
        raise ValueError("Autentifikasiya xətası")

    iv = nonce + (0).to_bytes(4, "big")
    return ctr_decrypt(ciphertext, enc_key, iv, aes_block_encrypt)

print("\n" + "-" * 70)
print("📦 AEAD KONSEPSİYASI")
print("-" * 70)
print("AEAD = Məxfilik + Tamlıq + AAD")
print("Bu hüceyrə generic Encrypt-then-MAC sxemini göstərir; GCM/CCM deyil.")


### 6.1 Generic composition üsulları

AEAD rejimləri iki kateqoriyaya bölünür:
1. **Daxili (dedicated) AEAD rejimləri:** GCM, CCM, ChaCha20-Poly1305
2. **Generic composition:** Şifrələmə və MAC-in kombinasiyası

Aşağıdakı cədvəl ümumi qaydanı göstərir: EtM standart fərziyyələr altında təhlükəsiz seçimdir, MtE və E&M isə protokol dizaynında səhvə daha həssasdır.

| Xüsusiyyət | Encrypt-then-MAC | MAC-then-Encrypt | Encrypt-and-MAC |
|------------|-------------------|-------------------|-----------------|
| CCA-təhlükəsizlik | ✅ CPA-təhlükəsiz şifrələmə, təhlükəsiz MAC, müstəqil açarlar və birmənalı kodlaşdırma ilə IND-CCA təhlükəsiz | ⚠️ Ümumən səhvə meyllidir; yalnız xüsusi şərtlərdə təhlükəsiz ola bilər | ⚠️ Ümumən səhvə meyllidir; yalnız xüsusi şərtlərdə təhlükəsiz ola bilər |
| Padding oracle hücumları | ✅ Düzgün yoxlama sırası ilə davamlı | ❌ Tarixi protokollarda həssas olub | ✅/⚠️ Konstruk­siyadan asılıdır |
| Paralel icra | ✅ Bəli | ❌ Xeyr | ✅ Bəli |
| Tətbiq nümunəsi | TLS 1.3-də AEAD yanaşması | SSL 3.0/TLS 1.0-da tarixi MtE | SSH-də tarixi E&M |


In [ ]:
def composition_demo():
    """
    Generic composition üsullarının nümayişi
    """
    print("\n" + "-" * 70)
    print("🔄 GENERIC COMPOSITION ÜSULLARI")
    print("-" * 70)

    print("""
    Encrypt-then-MAC (EtM):
      C = E_K1(P)
      T = MAC_K2(A || C)
      ✓ CCA təhlükəsiz
      ✓ TLS 1.3

    MAC-then-Encrypt (MtE):
      T = MAC_K2(A || P)
      C = E_K1(P || T)
      ✗ CCA təhlükəsiz DEYİL
      ✗ SSL 3.0, TLS 1.0

    Encrypt-and-MAC (E&M):
      C = E_K1(P)
      T = MAC_K2(A || P)
      ✓ SSH
    """)

composition_demo()

### ✍️ Çalışma 2: AEAD konsepsiyası (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **AAD nümunələri:** Əlaqəli verilənlərə (AAD) 3 nümunə verin.

2. **Generic composition:** Encrypt-then-MAC ilə MAC-then-Encrypt arasında nə fərq var?

3. **Nəzəri sual:** Niyə müasir protokollarda AEAD rejimləri məcburidir?

In [ ]:
# Çalışma 2 - Cavablar

print("📝 ÇALIŞMA 2 CAVABLARI")
print("=" * 80)

# 1. AAD nümunələri
print("\n1. AAD NÜMUNƏLƏRİ:")
print("""
   • IP paket başlığı (mənbə ünvanı, təyinat ünvanı, port nömrəsi)
   • Fayl metadatası (fayl adı, yaradılma tarixi, ölçü)
   • Protokol parametrləri (versiya nömrəsi, timestamp, nonce)
""")

# 2. Generic composition
print("\n2. GENERIC COMPOSITION:")
print("""
   Encrypt-then-MAC:
   • Əvvəl şifrlə, sonra MAC hesabla
   • CCA təhlükəsiz
   • TLS 1.3, IPsec

   MAC-then-Encrypt:
   • Əvvəl MAC hesabla, sonra şifrlə
   • Padding oracle hücumlarına qarşı həssas
   • SSL 3.0, TLS 1.0 (indi istifadə edilmir)
""")

# 3. AEAD rejimlərinin məcburiliyi
print("\n3. AEAD REJİMLƏRİNİN MƏCBURİLİYİ:")
print("""
   AEAD rejimləri məcburidir, çünki:
   • Məxfilik və tamlıq eyni anda tələb olunur
   • Ayrı-ayrı implementasiyalar səhv ola bilər
   • Padding oracle hücumlarının qarşısını alır
   • Daha sadə və təhlükəsiz protokol dizaynı
""")

## 🔐 GCM (Galois/Counter Mode) (30 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔐 GCM rejiminin quruluşu</h4>
<p>GCM rejimi iki komponentdən ibarətdir:</p>
<ul>
    <li><b>CTR rejimi</b> — məxfilik təmin edir</li>
    <li><b>GHASH funksiyası</b> — autentifikasiya təmin edir</li>
</ul>

<p style="text-align: center; font-size: 1.1em;">$$
\begin{align*}
H &= E_K(0^{128}) \\
Y_0 &= \begin{cases}
IV \parallel 0^{31}1 & \text{if } len(IV) = 96 \\
GHASH(H, \{\}, IV) & \text{otherwise}
\end{cases} \\
C_i &= P_i \oplus E_K(\text{incr}(Y_{i-1})) \\
T &= GHASH_H(A, C) \oplus E_K(Y_0)
\end{align*}
$$</p>
</div>

### 7.1 GHASH funksiyası və GF(2¹²⁸) arifmetikası

In [ ]:
def gf128_multiply(x, y):
    """
    GF(2^128) meydanında vurma (GCM/GHASH üçün düzgün bit sıralanması ilə).

    Reduksiya polinomu:
        x^128 + x^7 + x^2 + x + 1
    Implementasiya sabiti:
        R = 0xE1 << 120
    """
    r = 0xE1000000000000000000000000000000
    z = 0
    v = x

    for i in range(128):
        if (y >> (127 - i)) & 1:
            z ^= v

        if v & 1:
            v = (v >> 1) ^ r
        else:
            v >>= 1

    return z

def ghash(h, a, c):
    """
    GHASH funksiyası
    h: GHASH açarı (H = E_K(0^128))
    a: əlaqəli verilənlər (bytes)
    c: şifrəli mətn (bytes)
    """
    block_size = 16

    a_padded = a + b"\x00" * ((block_size - len(a) % block_size) % block_size)
    c_padded = c + b"\x00" * ((block_size - len(c) % block_size) % block_size)

    len_block = (len(a) * 8).to_bytes(8, "big") + (len(c) * 8).to_bytes(8, "big")

    x = 0
    blocks = a_padded + c_padded + len_block

    for i in range(0, len(blocks), block_size):
        block = int.from_bytes(blocks[i:i + block_size], "big")
        x = gf128_multiply(x ^ block, h)

    return x

print("\n" + "=" * 70)
print("🧮 GHASH FUNKSİYASI TESTİ")
print("=" * 70)

# Self-check: AES-GCM tag tənliyi ilə yoxlama
if CRYPTO_AVAILABLE:
    key = b"\x00" * 16
    nonce = b"\x00" * 12
    plaintext = b"\x00" * 16
    aad = b""

    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    cipher.update(aad)
    ciphertext, tag = cipher.encrypt_and_digest(plaintext)

    # XƏBƏRDARLIQ: ECB rejimi mesajları şifrələmək üçün təhlükəsiz deyil.
    # Burada yalnız GCM-in daxili tək-blok AES əməliyyatını hesablamaq üçün istifadə olunur:
    # H = E_K(0^128) və E_K(J0). Praktik şifrələmə üçün GCM/AEAD kimi rejimlərdən istifadə edin.
    H = int.from_bytes(AES.new(key, AES.MODE_ECB).encrypt(b"\x00" * 16), "big")
    J0 = nonce + (1).to_bytes(4, "big")
    ek_j0 = AES.new(key, AES.MODE_ECB).encrypt(J0)

    expected_s = int.from_bytes(tag, "big") ^ int.from_bytes(ek_j0, "big")
    computed_s = ghash(H, aad, ciphertext)

    print(f"Ciphertext: {ciphertext.hex()}")
    print(f"Tag:        {tag.hex()}")
    print(f"GHASH:      {computed_s:032x}")
    print(f"✅ Self-check uğurlu? {computed_s == expected_s}")
else:
    h = 0x66E94BD4EF8A2C3B884CFA59CA342B2E
    a = b"Hello"
    c = b"World"
    result = ghash(h, a, c)
    print(f"GHASH(H, A, C) = {result:032x}")

### 7.2 PyCryptodome ilə GCM

In [ ]:
def gcm_pycryptodome_demo():
    """
    PyCryptodome ilə GCM rejiminin nümayişi
    """
    if not CRYPTO_AVAILABLE:
        print("PyCryptodome yüklü deyil")
        return

    print("\n" + "=" * 80)
    print("🔐 GCM (GALOIS/COUNTER MODE) NÜMAYİŞİ")
    print("=" * 80)

    key = get_random_bytes(16)        # AES-128
    nonce = get_random_bytes(12)      # GCM üçün 96-bit nonce tövsiyə olunur

    plaintext = "Bu çox gizli mesajdır. Heç kim oxuya bilməz.".encode("utf-8")
    aad = "Əlaqəli məlumat (AAD) - şifrələnmir, amma tamlığı qorunur".encode("utf-8")

    print(f"🔑 Açar: {key.hex()}")
    print(f"🔢 Nonce: {nonce.hex()}")
    print(f"📨 Açıq mətn: {plaintext.decode('utf-8')}")
    print(f"📎 AAD: {aad.decode('utf-8')}")

    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    cipher.update(aad)
    ciphertext, tag = cipher.encrypt_and_digest(plaintext)

    print(f"\n🔒 Şifrəli mətn: {ciphertext.hex()}")
    print(f"🏷️ Autentifikasiya teqi: {tag.hex()}")

    decipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    decipher.update(aad)

    try:
        decrypted = decipher.decrypt_and_verify(ciphertext, tag)
        print(f"\n🔓 Deşifrələnmiş: {decrypted.decode('utf-8')}")
        print("✅ Autentifikasiya UĞURLU!")
    except ValueError:
        print("❌ Autentifikasiya XƏTASI! Məlumat dəyişdirilib")

    print("\n" + "-" * 50)
    print("AAD dəyişdirildikdə:")
    decipher2 = AES.new(key, AES.MODE_GCM, nonce=nonce)
    decipher2.update("Dəyişdirilmiş AAD".encode("utf-8"))

    try:
        decipher2.decrypt_and_verify(ciphertext, tag)
        print("❌ Bu işləməməlidir!")
    except ValueError:
        print("✅ Düzgün: AAD dəyişdirildikdə autentifikasiya xətası verir")

if CRYPTO_AVAILABLE:
    gcm_pycryptodome_demo()

### 7.3 GCM rejimində nonce təkrarı problemi

<div style="background-color: #ffebee; padding: 15px; border-radius: 10px; border-left: 5px solid #f44336;">
<h4>⚠️ GCM-də nonce təkrarı</h4>
<p>Eyni $(K, IV)$ cütü təkrar istifadə olunarsa:</p>
<ol>
    <li>Açar axını təkrarlanır ($C_1 \oplus C_2 = P_1 \oplus P_2$)</li>
    <li>GHASH açarı $H = E_K(0^{128})$ bərpa oluna bilər</li>
    <li>İxtiyari mesajlar üçün etibarlı teq yaratmaq mümkün olur</li>
</ol>
<p><b>Nəticə: Nonce təkrarı bütün təhlükəsizliyi çökdürür!</b></p>
</div>

In [ ]:
def gcm_nonce_reuse_demo():
    """
    GCM-də nonce təkrarının fəsadlarını göstərir
    """
    if not CRYPTO_AVAILABLE:
        return

    print("\n" + "-" * 80)
    print("⚠️ GCM NONCE TƏKRARI PROBLEMİ")
    print("-" * 80)

    key = get_random_bytes(16)
    nonce = b"\x00" * 12  # Sabit nonce (çox pis!)

    message1 = b"Message A - secret data 1234"
    message2 = b"Message B - secret data 5678"
    aad = b"header"

    print(f"📨 Mesaj 1: {message1}")
    print(f"📨 Mesaj 2: {message2}")
    print(f"📎 AAD: {aad}")

    cipher1 = AES.new(key, AES.MODE_GCM, nonce=nonce)
    cipher1.update(aad)
    c1, tag1 = cipher1.encrypt_and_digest(message1)

    cipher2 = AES.new(key, AES.MODE_GCM, nonce=nonce)
    cipher2.update(aad)
    c2, tag2 = cipher2.encrypt_and_digest(message2)

    print(f"\n🔒 c1: {c1.hex()}")
    print(f"🔒 c2: {c2.hex()}")

    xor_c = bytes(a ^ b for a, b in zip(c1, c2))
    xor_m = bytes(a ^ b for a, b in zip(message1, message2))

    print(f"\n🔢 c1 ⊕ c2: {xor_c.hex()}")
    print(f"🔢 m1 ⊕ m2: {xor_m.hex()}")
    print(f"\n✅ Bərabərdir? {xor_c == xor_m}")

    print("\n🚨 TƏHLÜKƏ: Nonce təkrarı bütün təhlükəsizliyi çökdürür!")
    print("   • Açar axını təkrarlanır")
    print("   • GHASH açarı eyni qalır")
    print("   • Saxta teq hücumları asanlaşır")

if CRYPTO_AVAILABLE:
    gcm_nonce_reuse_demo()

### ✍️ Çalışma 3: GCM rejimi (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **GCM testi:** PyCryptodome ilə GCM rejimində şifrləmə aparın. AAD dəyişdirildikdə nə baş verir?

2. **GHASH hesablanması:** GHASH funksiyasını $H = 0x66E94BD4EF8A2C3B884CFA59CA342B2E$ üçün hesablayın:
   * $A = \text{0x01}$, $C = \text{0x02}$

3. **Nonce təkrarı:** GCM-də nonce təkrarı niyə təhlükəlidir? Riyazi izah verin.

4. **Maksimum mesaj:** GCM rejimində eyni $(K, IV)$ cütü ilə maksimum nə qədər məlumat şifrləmək olar?

In [ ]:
# Çalışma 3 - Cavablar

print("📝 ÇALIŞMA 3 CAVABLARI")
print("=" * 80)

# 1. GCM testi (artıq yuxarıda edildi)
print("\n1. GCM TESTİ:")
print("   GCM-də AAD dəyişdirildikdə autentifikasiya xətası verir.")
print("   Bu, məlumat bütövlüyünü təmin edir.")

# 2. GHASH hesablanması
print("\n2. GHASH HESABLANMASI:")
h = 0x66E94BD4EF8A2C3B884CFA59CA342B2E
a = b'\x01'
c = b'\x02'
ghash_result = ghash(h, a, c)
print(f"   GHASH(H, A, C) = {ghash_result:032x}")

# 3. Nonce təkrarı
print("\n3. NONCE TƏKRARI:")
print("""
   GCM-də nonce təkrarı təhlükəlidir, çünki:

   • CTR hissə: c1 ⊕ c2 = m1 ⊕ m2 (açar axını təkrarlanır)
   • GHASH hissə: H = E_K(0^128) eynidir
   • İki mesajın teqləri arasında əlaqə: T1 ⊕ T2 = GHASH_H(A1, C1) ⊕ GHASH_H(A2, C2)
   • H bərpa oluna bilər → ixtiyari teq yaratmaq mümkün

   Nəticə: Nonce təkrarı bütün təhlükəsizliyi çökdürür!
""")

# 4. Maksimum mesaj
print("\n4. MAKSİMUM MESAJ:")
print("""
   GCM-də bir (K, IV) üçün maksimum açıq mətn uzunluğu:
   len(P) ≤ 2^39 - 256 bit, yəni ən çox 2^32 - 2 AES bloku.

   Səbəblər:
   • GCM-də 32-bit sayğac sahəsinin bütün qiymətləri məlumat blokları üçün istifadə edilə bilməz
   • Hər AES bloku 16 baytdır → (2^32 - 2) * 16 = 64 GiB - 32 bayt
   • Daha çox şifrləmə sayğac təkrarına səbəb ola bilər

   Tövsiyə: Bir (K, IV) ilə 2^39 - 256 bit limitini aşmayın.
""")


## 🛡️ GCM-SIV: Nonce Təkrarına Davamlı AEAD (15 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🛡️ GCM-SIV rejiminin iş prinsipi</h4>
<p>GCM-SIV rejimi klassik GCM-dən fərqli olaraq <b>sintetik IV/teqi əvvəlcə hesablayır</b>:</p>

<ol>
    <li>Ayrıca autentifikasiya və şifrələmə açarları törədilir.</li>
    <li>$SIV = \text{POLYVAL}_{H}(\text{nonce}, A, P, \text{uzunluqlar})$ kimi yüksək səviyyəli şəkildə sintetik IV/teq hesablanır.</li>
    <li>$C$ sintetik IV-dən törədilmiş counter mode keystream ilə alınır.</li>
</ol>

<p><b>Üstünlüklər:</b></p>
<ul>
    <li>✅ Nonce təkrarına davamlı</li>
    <li>✅ Deterministik şifrələmə (eyni P, A, IV → eyni C, T)</li>
    <li>✅ Standartlaşdırılmış konstruksiya (RFC 8452)</li>
</ul>

<p>RFC 8452 ilə standartlaşdırılıb.</p>
</div>

In [ ]:
def gcm_siv_concept_demo():
    """
    GCM-SIV rejiminin konsepsiyası (sadələşdirilmiş, amma daha düzgün izah)
    """
    print("\n" + "-" * 80)
    print("🛡️ GCM-SIV: NONCE TƏKRARINA DAVAMLI AEAD")
    print("-" * 80)

    print("""
    GCM-SIV rejiminin əsas ideyası:

    1. Əvvəl sintetik IV (SIV) hesablanır.
       Bu addımda plaintext, AAD və nonce birlikdə emal olunur.

    2. GCM-dən fərqli olaraq autentifikator GHASH deyil, POLYVAL ailəsinə əsaslanır.

    3. Sonra plaintext CTR tipli axınla şifrlənir və bu axın SIV-dən törədilir.

    4. Nonce təkrarlandıqda rejim "misuse-resistant" qalır:
       eyni girişlər eyni çıxışı verə bilər, amma GCM-dəki kimi dərhal
       məxfilik və tamlıq fəlakəti baş vermir.

    Üstünlüklər:
      • Təsadüfi nonce təkrarına daha davamlıdır
      • Səhv nonce idarəsi olan sistemlər üçün daha təhlükəsizdir

    Çatışmazlıq:
      • Şifrləmə adətən iki keçid tələb edir və GCM-dən yavaşdır
    """)

gcm_siv_concept_demo()

### ✍️ Çalışma 4: GCM-SIV (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **GCM vs GCM-SIV:** GCM-SIV rejiminin klassik GCM rejiminə nisbətən üstünlüyü nədir?

2. **Teq hesablanması:** GCM-SIV-də teq əvvəlcə hesablanır. Bu, performansa necə təsir edir?

3. **Praktik tətbiq:** Hansı protokollarda GCM-SIV istifadə olunur?

In [ ]:
# Çalışma 4 - Cavablar

print("📝 ÇALIŞMA 4 CAVABLARI")
print("=" * 80)

# 1. GCM vs GCM-SIV
print("\n1. GCM vs GCM-SIV:")
print("""
   GCM-SIV-in üstünlükləri:
   • Nonce təkrarına daha davamlıdır
   • Eyni (nonce, AAD, plaintext) üçlüyü üçün deterministik çıxış verir
   • Təsadüfi nonce reuse zamanı GCM kimi dərhal fəlakət yaşamır

   GCM-in üstünlükləri:
   • Daha sadə və daha sürətlidir
   • Bir çox platformada geniş aparat sürətləndirilməsi var
""")

# 2. Teq hesablanması
print("\n2. TEQ HESABLANMASI:")
print("""
   GCM-SIV-də əvvəl sintetik IV hesablandığı üçün:
   • Şifrləmə iki keçid tələb edir
   • GCM-dən yavaşdır
   • Bunun əvəzində nonce reuse riskinə qarşı daha dözümlüdür
""")

# 3. Praktik tətbiq
print("\n3. PRAKTİK TƏTBİQ:")
print("""
   GCM-SIV daha çox aşağıdakı hallarda məntiqlidir:
   • nonce idarəsinin çətin olduğu sistemlər
   • birdən çox göndərənin eyni açarı paylaşdığı arxitekturalar
   • vəziyyətin (state) itə biləcəyi embedded/offline sistemlər

   Qeyd: GCM-SIV standart TLS 1.3 cipher suite siyahısının bir hissəsi deyil.
""")

## 📦 CCM (Counter with CBC-MAC) Rejimi (15 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>📦 CCM rejiminin quruluşu</h4>
<p>CCM rejimi CTR rejimi və CBC-MAC-in birləşməsindən yaranır:</p>

<ol>
    <li>$\text{T} = \text{CBC-MAC}_{K}(A \parallel P \parallel \text{lengths})$</li>
    <li>$C = P \oplus \text{CTR}_{K}(IV)$</li>
    <li>$T_{\text{enc}} = T \oplus \text{CTR}_{K}(IV_0)$</li>
</ol>

<p><b>Xüsusiyyətlər:</b></p>
<ul>
    <li>IEEE 802.11i (WPA2) standartı</li>
    <li>CBC-MAC paralel deyil → məhdud performans</li>
    <li>Maksimum mesaj uzunluğu sabit deyil: $q = 15 - \text{nonce\_length}$ olduqda payload uzunluğu $p < 2^{8q}$ bayt ilə məhdudlaşır</li>
</ul>
</div>

In [ ]:
def ccm_demo():
    """
    CCM rejiminin nümayişi (cryptography kitabxanası ilə)
    """
    if not CRYPTOGRAPHY_AVAILABLE:
        print("cryptography yüklü deyil")
        return

    print("\n" + "=" * 80)
    print("📦 CCM (COUNTER WITH CBC-MAC) REJİMİ")
    print("=" * 80)

    from cryptography.hazmat.primitives.ciphers.aead import AESCCM

    key = AESCCM.generate_key(bit_length=128)
    nonce = os.urandom(13)  # 13-bayt nonce => q=2 => max payload 65535 bayt

    plaintext = "WPA2 Wi-Fi şəbəkələrində istifadə olunur".encode("utf-8")
    aad = "əlaqəli verilənlər".encode("utf-8")

    print(f"🔑 Açar: {key.hex()}")
    print(f"🔢 Nonce: {nonce.hex()}")
    print(f"📨 Açıq mətn: {plaintext.decode('utf-8')}")
    print(f"📎 AAD: {aad.decode('utf-8')}")

    ccm = AESCCM(key)
    ciphertext = ccm.encrypt(nonce, plaintext, aad)

    print(f"\n🔒 Şifrəli mətn (cəmi {len(ciphertext)} bayt): {ciphertext.hex()}")

    try:
        decrypted = ccm.decrypt(nonce, ciphertext, aad)
        print(f"🔓 Deşifrələnmiş: {decrypted.decode('utf-8')}")
        print("✅ Autentifikasiya UĞURLU!")
    except Exception:
        print("❌ Autentifikasiya XƏTASI!")

if CRYPTOGRAPHY_AVAILABLE:
    ccm_demo()

### 9.1 GCM və CCM müqayisəsi

In [ ]:
def gcm_vs_ccm_comparison():
    """
    GCM və CCM rejimlərinin müqayisəsi
    """
    print("\n" + "-" * 80)
    print("📊 GCM vs CCM MÜQAYİSƏSİ")
    print("-" * 80)

    print("""
    | Xüsusiyyət        | GCM                                      | CCM                                      |
    |-------------------|------------------------------------------|------------------------------------------|
    | Əsas mexanizm      | CTR + GHASH                              | CTR + CBC-MAC                            |
    | Paralel icra       | ✅ Bəli                                  | ❌ Xeyr                                  |
    | Performans         | ✅ Yüksək                                | ⚠️ Adətən daha yavaş                     |
    | IV / nonce         | 96-bit tövsiyə olunur                    | 7-13 bayt                                |
    | Maksimum mesaj     | len(P) ≤ 2^39 - 256 bit                  | nonce uzunluğundan asılıdır              |
    | Nümunə             | ≈ 64 GiB plaintext limiti                | 13-bayt nonce ilə max 65535 bayt         |
    | Tətbiqlər          | TLS 1.3, IPsec                           | IEEE 802.11i / CCMP                      |
    | Aparat dəstəyi     | AES-NI, PCLMULQDQ                        | AES-NI                                   |
    """)

gcm_vs_ccm_comparison()

### ✍️ Çalışma 5: CCM rejimi (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **CCM vs GCM:** CCM rejiminin GCM rejiminə nisbətən əsas çatışmazlıqlarını sadalayın.

2. **Praktik tətbiq:** Hansı protokol CCM rejimindən istifadə edir?

3. **Maksimum mesaj:** CCM rejimində maksimum mesaj uzunluğu nə qədərdir? Niyə?

In [ ]:
# Çalışma 5 - Cavablar

print("📝 ÇALIŞMA 5 CAVABLARI")
print("=" * 80)

# 1. CCM vs GCM
print("\n1. CCM vs GCM:")
print("""
   CCM rejiminin çatışmazlıqları:
   • CBC-MAC paralel deyil → adətən daha yavaşdır
   • Maksimum mesaj ölçüsü nonce uzunluğundan asılıdır
   • Nonce təkrarı təhlükəlidir

   Üstünlükləri:
   • Yaxşı standartlaşdırılıb
   • Wi‑Fi CCMP kimi praktik istifadəsi var
""")

# 2. Praktik tətbiq
print("\n2. PRAKTİK TƏTBİQ:")
print("""
   CCM rejimi IEEE 802.11i (WPA2) standartında istifadə olunur:
   • Wi‑Fi şəbəkələrində CCMP
   • Məhdud resurslu sistemlər üçün münasib seçim ola bilər
""")

# 3. Maksimum mesaj
print("\n3. MAKSİMUM MESAJ:")
print("""
   CCM-də maksimum mesaj uzunluğu sabit 32 KB deyil.

   O, q parametrindən (və dolayısı ilə nonce uzunluğundan) asılıdır:
   • q ∈ {2,3,4,5,6,7,8}
   • p < 2^(8q) bayt
   • 13-bayt nonce üçün q=2 olduğuna görə maksimum payload = 65535 baytdır
""")

## 💽 XTS Rejimi: Disk Şifrələməsi (15 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>💽 Disk şifrələməsinin tələbləri</h4>
<ol>
    <li><b>Təsadüfi giriş:</b> İstənilən sektoru digərlərindən asılı olmadan şifrləmək</li>
    <li><b>Eyni məlumat fərqli görünməlidir:</b> Müxtəlif sektorlarda fərqli şifrəli mətn</li>
    <li><b>Genişlənmə yox:</b> Şifrəli mətn açıq mətnlə eyni ölçüdə</li>
    <li><b>Yüksək performans:</b> Disk sürətinə çatmaq üçün</li>
</ol>
</div>

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4; margin-top: 10px;">
<h4>💽 XTS rejiminin quruluşu</h4>
<p>XTS rejimi iki açar istifadə edir:</p>
<ul>
    <li>$K_1$ — məlumatı şifrləmək üçün</li>
    <li>$K_2$ — tweak dəyərini şifrləmək üçün</li>
</ul>

<p style="text-align: center; font-size: 1.1em;">$$C = E_{K_1}(P \oplus X) \oplus X$$</p>
<p style="text-align: center; font-size: 1.1em;">$$X = E_{K_2}(i) \otimes \alpha^j$$</p>
</div>

In [ ]:
XTS_DEMO_KEY2 = b"XTS-demo-key-123"

def gf128_mul_alpha_xts(tweak):
    """
    XTS üçün tweak-i α = x (yəni 2) ilə vurur.
    0x87 burada reduksiya sabitidir, α-nın özü deyil.
    """
    t = bytearray(tweak)
    carry = 0

    for i in range(16):
        new_carry = (t[i] >> 7) & 1
        t[i] = ((t[i] << 1) & 0xFF) | carry
        carry = new_carry

    if carry:
        t[0] ^= 0x87

    return bytes(t)

def xts_tweak_calculation(sector_number, block_index, key2=XTS_DEMO_KEY2):
    """
    XTS tweak hesablanması.

    T_0 = AES-ECB_{K2}(sector_number_128_le)
    T_i = α * T_{i-1}

    Qeyd:
      • Bu funksiya tweak hesablayır; tam XTS şifrləmə deyil.
      • Sektor nömrəsi 128-bit little-endian kimi kodlanır.
      • AES-ECB burada yalnız XTS standartında T_0 tweak dəyərini hesablamaq
        üçün istifadə olunur. Ümumi məlumat şifrləməsi üçün ECB rejimi
        semantik təhlükəsiz deyil və istifadə edilməməlidir.
    """
    if not CRYPTO_AVAILABLE:
        raise RuntimeError("PyCryptodome tələb olunur")
    if sector_number < 0 or block_index < 0:
        raise ValueError("sector_number və block_index mənfi ola bilməz")
    if len(key2) not in {16, 32}:
        raise ValueError("K2 açarı 16 və ya 32 bayt olmalıdır")

    tweak_input = sector_number.to_bytes(16, "little")
    # Xəbərdarlıq: ECB rejimi ümumi plaintext bloklarını şifrləmək üçün təhlükəsiz
    # deyil. Burada yalnız XTS tweak hesablamasında tək 128-bit girişə AES tətbiq edilir.
    tweak = AES.new(key2, AES.MODE_ECB).encrypt(tweak_input)

    for _ in range(block_index):
        tweak = gf128_mul_alpha_xts(tweak)

    return tweak

print("\n" + "=" * 70)
print("💽 XTS TWEAK HESABLANMASI")
print("=" * 70)

sector = 42
for i in range(5):
    tweak = xts_tweak_calculation(sector, i)
    print(f"Sektor {sector}, blok {i}: tweak = {tweak.hex()}")

### 10.1 XTS rejiminin tətbiqləri

In [ ]:
def xts_applications_demo():
    """
    XTS rejiminin tətbiqləri
    """
    print("\n" + "-" * 70)
    print("📌 XTS TƏTBİQLƏRİ")
    print("-" * 70)

    print("""
    XTS rejimi əsasən disk və blok qurğusu şifrələməsində istifadə olunur:

    • Windows BitLocker
    • Linux dm-crypt / LUKS
    • Apple ekosistemində yaddaş şifrələmə ssenariləri
    • Müxtəlif SSD / disk controller həlləri

    Qeyd:
    • XTS AEAD deyil
    • Məqsədi məlumatın konfidensiallığını qorumaqdır
    • Tamlıq / autentifikasiya ayrıca mexanizmlə təmin olunmalıdır
    """)

xts_applications_demo()

### ✍️ Çalışma 6: XTS rejimi (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Tweak hesablanması:** XTS rejimində sektor nömrəsi 5 olan bir sektorun 3-cü blokunu şifrləmək üçün istifadə olunan $X$ dəyərini ifadə edin. $\alpha = 2$ qəbul edin.

2. **XTS vs CBC:** XTS rejimi ilə CBC rejimi arasında hansı əsas fərqlər var?

3. **Praktik tətbiq:** Hansı əməliyyat sistemləri XTS rejimindən istifadə edir?

In [ ]:
# Çalışma 6 - Cavablar

print("📝 ÇALIŞMA 6 CAVABLARI")
print("=" * 80)

# 1. Tweak hesablanması
print("\n1. TWEAK HESABLANMASI:")
print("""
   Sektor 5, blok 3 üçün:

   T_0 = AES-ECB_{K2}(encode_le_128(5))
   T_1 = α · T_0
   T_2 = α · T_1
   T_3 = α · T_2

   Burada α = x (praktik implementasiyada "bir bit sola sürüşdür" kimi görünür),
   0x87 isə daşma zamanı reduksiya sabitidir.
""")

# 2. XTS vs CBC
print("\n2. XTS vs CBC:")
print("""
   XTS-in CBC-dən fərqləri:
   • Hər blok tweak ilə şifrlənir
   • Təsadüfi giriş və blok-sektor əsaslı işləmə üçün münasibdir
   • Disk şifrələməsi üçün dizayn olunub
   • İki AES açarı istifadə edir

   CBC disk üçün əlverişsizdir:
   • Zəncirvari asılılıq yaradır
   • Təsadüfi giriş çətinləşir
""")

# 3. Praktik tətbiq
print("\n3. PRAKTİK TƏTBİQ:")
print("""
   XTS istifadə sahələri:
   • BitLocker
   • dm-crypt / LUKS
   • müxtəlif storage-controller və disk şifrələmə həlləri
""")

## 🌐 Müasir Protokollarda AEAD Rejimləri (15 dəq)

In [ ]:
def tls13_aead_demo():
    """
    TLS 1.3-də AEAD rejimlərinin istifadəsi
    """
    print("\n" + "=" * 80)
    print("🌐 TLS 1.3 VƏ AEAD REJİMLƏRİ")
    print("=" * 80)

    print("""
    TLS 1.3-də müəyyən edilmiş əsas cipher suite-lər:
    • TLS_AES_128_GCM_SHA256
    • TLS_AES_256_GCM_SHA384
    • TLS_CHACHA20_POLY1305_SHA256
    • TLS_AES_128_CCM_SHA256
    • TLS_AES_128_CCM_8_SHA256

    Uyğunluq baxımından:
    • AES-128-GCM məcburi implementasiyadır
    • AES-256-GCM və ChaCha20-Poly1305 SHOULD səviyyəsində tövsiyə olunur
    • CCM variantları da TLS 1.3-də müəyyən edilib, amma daha az yayğındır

    Xüsusiyyətlər:
    • Hər TLS rekordu AEAD ilə qorunur
    • AAD: rekord başlığı (5 bayt)
    • Nonce: statik IV ilə 64-bit sıra nömrəsinin XOR-u nəticəsində alınan per-record nonce
    """)

tls13_aead_demo()

In [ ]:
def wireguard_chacha20_demo():
    """
    WireGuard VPN-də ChaCha20-Poly1305 AEAD rejimi
    """
    if not CRYPTOGRAPHY_AVAILABLE:
        return

    print("\n" + "-" * 80)
    print("🔌 WIREGUARD VƏ CHACHA20-POLY1305")
    print("-" * 80)

    from cryptography.hazmat.primitives.ciphers.aead import ChaCha20Poly1305

    key = ChaCha20Poly1305.generate_key()

    # WireGuard-a yaxın nümunə: 64-bit sayğac + 32 sıfır bit => 96-bit nonce
    packet_counter = 1
    nonce = b"\x00" * 4 + packet_counter.to_bytes(8, "little")

    plaintext = b"WireGuard VPN paketi - cox gizli melumat"
    aad = b"paket basligi (IP, port, timestamp)"

    print(f"🔑 Açar: {key.hex()}")
    print(f"🔢 Nonce: {nonce.hex()} (counter={packet_counter})")
    print(f"📨 Açıq mətn: {plaintext}")
    print(f"📎 AAD: {aad}")

    chacha = ChaCha20Poly1305(key)
    ciphertext = chacha.encrypt(nonce, plaintext, aad)

    print(f"\n🔒 Şifrəli mətn (cəmi {len(ciphertext)} bayt): {ciphertext.hex()}")

    try:
        decrypted = chacha.decrypt(nonce, ciphertext, aad)
        print(f"🔓 Deşifrələnmiş: {decrypted}")
        print("✅ Autentifikasiya UĞURLU!")
    except Exception:
        print("❌ Autentifikasiya XƏTASI!")

if CRYPTOGRAPHY_AVAILABLE:
    wireguard_chacha20_demo()

### ✍️ Çalışma 7: Müasir protokollar (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **TLS 1.3:** TLS 1.3-də hansı AEAD rejimləri dəstəklənir?

2. **WireGuard:** WireGuard VPN-də niyə məhz ChaCha20-Poly1305 istifadə olunur?

3. **WPA3:** WPA3-Enterprise 192-bit və bəzi Wi‑Fi 6/7 ssenarilərində GCMP/GCMP-256 niyə seçilir? CCMP-128 hansı WPA3 profillərində hələ istifadə oluna və ya icazəli qala bilər?

In [ ]:
# Çalışma 7 - Cavablar

print("📝 ÇALIŞMA 7 CAVABLARI")
print("=" * 80)

# 1. TLS 1.3
print("\n1. TLS 1.3 AEAD REJİMLƏRİ:")
print("""
   TLS 1.3 üçün müəyyən edilmiş AEAD suite-lər:
   • AES-128-GCM
   • AES-256-GCM
   • ChaCha20-Poly1305
   • AES-128-CCM
   • AES-128-CCM_8

   Uyğunluq baxımından:
   • AES-128-GCM — MUST implement
   • AES-256-GCM və ChaCha20-Poly1305 — SHOULD implement
""")

# 2. WireGuard
print("\n2. WIREGUARD VƏ CHACHA20-POLY1305:")
print("""
   WireGuard-da ChaCha20-Poly1305 seçilməsinin səbəbləri:
   • Geniş platforma dəstəyi
   • AES sürətləndirilməsi olmayan cihazlarda da sürətlidir
   • Sadə və təhlükəsiz proqram implementasiyası mümkündür
   • Nonce packet counter əsasında qurulur
""")

# 3. WPA3
print("\n3. WPA3 VƏ GCMP:")
print("""
   Wi‑Fi ekosistemində GCM əsaslı GCMP yüksək performanslı ssenarilər üçün cəlbedicidir:
   • paralel emal
   • yüksək ötürmə sürəti
   • müasir prosessorlarda yaxşı sürətləndirmə

   Qeyd: Wi‑Fi ailəsində CCMP də hələ mühüm yer tutur; GCMP bütün hallarda sadəcə "tam əvəz" deyil.
""")

## 📊 AEAD Rejimlərinin Təhlükəsizlik Müqayisəsi (5 dəq)

In [ ]:
def aead_comparison():
    """
    AEAD rejimlərinin müqayisəsi
    """
    print("\n" + "=" * 100)
    print("📊 AEAD REJİMLƏRİNİN MÜQAYİSƏSİ")
    print("=" * 100)

    comparison = """
┌────────────────────┬────────────┬─────────────┬──────────────────┬────────────────────┐
│ Xüsusiyyət         │ GCM        │ GCM-SIV     │ CCM              │ ChaCha20-Poly1305  │
├────────────────────┼────────────┼─────────────┼──────────────────┼────────────────────┤
│ Nonce təkrarı      │ ❌ Fəlakət │ ✅ Davamlı  │ ❌ Fəlakət       │ ❌ Fəlakət         │
│ Paralel şifrləmə   │ ✅ Bəli    │ ✅ Bəli     │ ❌ Xeyr          │ ✅ Bəli            │
│ Təsadüfi giriş     │ ✅ Bəli    │ ✅ Bəli     │ ❌ Xeyr          │ ✅ Bəli            │
│ Maksimum mesaj     │ ≈ 64 GiB   │ istifadədən │ nonce-dən asılı  │ böyük praktik limit │
│                    │            │ asılı limit │                  │                    │
│ Əsas tətbiq        │ TLS, IPsec │ misuse-     │ CCMP / WPA2      │ TLS, WireGuard     │
│                    │            │ resistant   │                  │                    │
│                    │            │ dizaynlar   │                  │                    │
└────────────────────┴────────────┴─────────────┴──────────────────┴────────────────────┘
"""
    print(comparison)
    print("Qeyd: XTS AEAD deyil; o, disk şifrələməsi üçün tweakable confidentiality mode-dur.")

aead_comparison()

## 🔎 Audit yekunu

Bu notebook düzəldilərkən aşağıdakı əsas problemlər aradan qaldırıldı:

- Python-da işləməyən `b"..."` qeyri-ASCII byte literal-lar UTF-8 `encode()` ilə düzəldildi.
- CTR nümunəsi real AES blok primitivindən istifadə edəcək şəkildə yeniləndi.
- GHASH vurması GCM üçün düzgün GF(2^128) alqoritmi ilə əvəz edildi.
- CCM üzrə maksimum mesaj ölçüsü barədə səhv `32 KB` iddiası düzəldildi.
- XTS hissəsi sadə int oyuncağından daha düzgün tweak hesabına çevrildi.
- TLS 1.3, GCM-SIV və WireGuard barədə bəzi faktiki qeyri-dəqiqliklər düzəldildi.
- XTS-in AEAD olmadığı ayrıca qeyd edildi.

## ✅ Praktik Tövsiyələr (10 dəq)

<div style="background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-left: 5px solid #4caf50;">
<h4>✅ AEAD rejimlərinin düzgün implementasiyası üçün tövsiyələr</h4>

<p><b>1. Nonce idarəetməsi:</b></p>
<ul>
    <li>96-bit IV istifadə edin (GCM üçün)</li>
    <li>Heç vaxt eyni $(K, IV)$ cütünü təkrar istifadə etməyin</li>
    <li>Sayğac əsaslı nonce-lar təhlükəsizdir</li>
</ul>

<p><b>2. Teq uzunluğu:</b></p>
<ul>
    <li>Minimum 96-bit teq istifadə edin</li>
    <li>128-bit teq tövsiyə olunur</li>
</ul>

<p><b>3. Maksimum mesaj uzunluğu:</b></p>
<ul>
    <li>GCM üçün: $2^{32}$ blokdan (64 GB) çox istifadə etməyin</li>
    <li>CCM üçün: 32 KB-dan çox istifadə etməyin</li>
</ul>

<p><b>4. Açar idarəetməsi:</b></p>
<ul>
    <li>Hər sessiya üçün yeni açar yaradın</li>
    <li>Açar rotasiyası tətbiq edin</li>
</ul>
</div>

<div style="background-color: #ffebee; padding: 15px; border-radius: 10px; border-left: 5px solid #f44336; margin-top: 10px;">
<h4>⚠️ ÖLÜMCÜL XƏTALAR</h4>
<ul>
    <li><b>Nonce təkrarı</b> - GCM, CCM, ChaCha20-də fəlakətdir</li>
    <li><b>Qısa teq</b> - 64-bit teq təhlükəsiz deyil</li>
    <li><b>Açar təkrarı</b> - eyni açar çox istifadə edilməməlidir</li>
    <li><b>AAD-nin yoxlanılmaması</b> - bütövlük təmin edilmir</li>
</ul>
</div>

## 🖥️ İnteqrasiya edilmiş tətbiq (15 dəq)

In [ ]:
def aead_lab_menu():
    """
    AEAD laboratoriyası interaktiv menyu
    """
    while True:
        print("\n" + "=" * 70)
        print("🔐 AEAD LABORATORİYASI - Məşğələ 12b")
        print("=" * 70)
        print("1. 🔢 CTR rejimi nümayişi")
        print("2. 🔐 GCM rejimi (PyCryptodome)")
        print("3. ⚠️ GCM nonce təkrarı problemi")
        print("4. 📦 CCM rejimi (WPA2)")
        print("5. 🔌 ChaCha20-Poly1305 (WireGuard)")
        print("6. 💽 XTS tweak hesablanması")
        print("7. 📊 Rejim müqayisəsi")
        print("8. 🧮 GHASH funksiyası testi")
        print("0. ❌ Çıxış")
        print("=" * 70)

        choice = input("📌 Seçiminiz: ")

        if choice == "1":
            plain = input("Mətn: ").encode("utf-8")
            key = get_random_bytes(16) if CRYPTO_AVAILABLE else os.urandom(16)
            nonce = get_random_bytes(12) if CRYPTO_AVAILABLE else os.urandom(12)
            iv = nonce + (0).to_bytes(4, "big")

            cipher = ctr_encrypt(plain, key, iv, aes_block_encrypt)
            dec = ctr_decrypt(cipher, key, iv, aes_block_encrypt)

            print(f"Nonce: {nonce.hex()}")
            print(f"Şifrəli: {cipher.hex()}")
            print(f"Deşifrə: {dec.decode('utf-8', errors='replace')}")

        elif choice == "2" and CRYPTO_AVAILABLE:
            plain = input("Mətn: ").encode("utf-8")
            aad = input("AAD: ").encode("utf-8")

            key = get_random_bytes(16)
            nonce = get_random_bytes(12)

            cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
            cipher.update(aad)
            c, tag = cipher.encrypt_and_digest(plain)

            print(f"Şifrəli: {c.hex()}")
            print(f"Teq: {tag.hex()}")

            decipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
            decipher.update(aad)
            try:
                dec = decipher.decrypt_and_verify(c, tag)
                print(f"Deşifrə: {dec.decode('utf-8', errors='replace')}")
                print("✅ Autentifikasiya uğurlu")
            except Exception:
                print("❌ Autentifikasiya xətası")

        elif choice == "3" and CRYPTO_AVAILABLE:
            gcm_nonce_reuse_demo()

        elif choice == "4" and CRYPTOGRAPHY_AVAILABLE:
            ccm_demo()

        elif choice == "5" and CRYPTOGRAPHY_AVAILABLE:
            wireguard_chacha20_demo()

        elif choice == "6":
            sector = int(input("Sektor nömrəsi: "))
            block = int(input("Blok indeksi: "))
            tweak = xts_tweak_calculation(sector, block)
            print(f"Tweak dəyəri: {tweak.hex()}")

        elif choice == "7":
            aead_comparison()

        elif choice == "8":
            h = int(input("H (hex): ") or "66e94bd4ef8a2c3b884cfa59ca342b2e", 16)
            a = input("A (bytes): ").encode("utf-8")
            c = input("C (bytes): ").encode("utf-8")
            result = ghash(h, a, c)
            print(f"GHASH = {result:032x}")

        elif choice == "0":
            print("👋 Proqram bitdi. Sağ olun!")
            break

        else:
            print("❌ Yanlış seçim")

# Proqramı işə sal (istəyə bağlı)
# aead_lab_menu()


## 🏠 Ev tapşırığı

### 📦 Ev tapşırığı 1: AEAD paketi (3 bal)

Aşağıdakı funksiyaları özündə birləşdirən Python paketi yazın:

```
aead_package/
├── __init__.py
├── ctr.py                 # CTR rejimi (paralel versiya)
├── ghash.py               # GHASH funksiyası, GF(2^128) arifmetikası
├── gcm.py                 # GCM şifrləmə/deşifrləmə
├── ccm.py                 # CCM rejimi (CTR + CBC-MAC)
├── xts.py                 # XTS tweak hesablanması
├── aead.py                # AEAD şifrləmə/deşifrləmə
└── main.py                # İnteraktiv menyu
```

**Tələblər:**
* Hər bir funksiya üçün docstring yazın
* Səhv hallarını idarə edin (try-except)
* Kod təmiz və oxunaqlı olmalıdır
* Hər modul üçün test nümunələri əlavə edin

### 🔐 Ev tapşırığı 2: Praktik məsələlər (2 bal)

Aşağıdakı məsələləri həll edin:

1. **CTR paralel şifrləmə:** CTR rejimində paralel şifrləmə implementasiya edin (threading istifadə edin). 100 MB məlumat üçün sürət testi aparın.

2. **GHASH implementasiyası:** GHASH funksiyasını implementasiya edin və test vektorları ilə yoxlayın.

3. **GCM nonce təkrarı:** GCM rejimində nonce təkrarı problemini simulyasiya edin. c1 ⊕ c2 = m1 ⊕ m2 olduğunu göstərin.

4. **ChaCha20-Poly1305 testi:** ChaCha20-Poly1305 AEAD rejimini cryptography kitabxanası ilə test edin.

### 📚 Ev tapşırığı 3: Tədqiqat (2 bal)

Araşdırma aparın və aşağıdakı suallara cavab tapın. Cavablarınızı 1-2 səhifəlik hesabat şəklində təqdim edin:

1. **NIST SP 800-38D:** NIST SP 800-38D sənədində GCM rejimi haqqında nə deyilir?
   * Təhlükəsizlik limitləri
   * IV uzunluğu tövsiyələri
   * Teq uzunluğu tövsiyələri

2. **TLS 1.3 rejimləri:** TLS 1.3-də niyə AES-128-GCM mandatory-to-implement, AES-256-GCM və ChaCha20-Poly1305 isə SHOULD səviyyəsində tövsiyə olunur?
   * CBC rejiminin problemləri (padding oracle)
   * Performans müqayisələri
   * Hardware dəstəyi

3. **GCM-SIV:** GCM-SIV rejimi ilə klassik GCM arasında əsas fərqlər nələrdir?
   * Nonce təkrarına davamlılıq
   * Performans fərqi
   * Tətbiq sahələri

4. **Disk şifrləməsi:** Disk şifrləməsində XTS rejiminin üstünlükləri nələrdir?
   * CBC-nin problemləri
   * Tweak anlayışı
   * Məlumat miqrasiyası

**Format tələbləri:**
* PDF formatında təqdim edin
* Ən azı 5 qaynaq göstərin (kitab, məqalə, veb səhifə)
* Öz sözlərinizlə yazın (copy-paste yox)
* Mümkünsə, qrafiklər və ya diaqramlar əlavə edin


## 📌 Yekun və müzakirə sualları

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; border-left: 5px solid #2c3e50;">
<h3>📋 Xülasə</h3>
<p>Bu məşğələdə öyrəndiklərimiz:</p>
<ul>
    <li>✅ <b>CTR rejimi</b>: Paralel şifrləmə, təsadüfi giriş, nonce təkrarı təhlükəsi</li>
    <li>✅ <b>AEAD</b>: Məxfilik + tamlıq + autentifikasiya, əlaqəli verilənlər (AAD)</li>
    <li>✅ <b>GCM</b>: CTR + GHASH, yüksək sürət, TLS 1.3-də standart</li>
    <li>✅ <b>GCM-SIV</b>: Nonce təkrarına davamlı, RFC 8452</li>
    <li>✅ <b>CCM</b>: CTR + CBC-MAC, WPA2-də istifadə olunur</li>
    <li>✅ <b>XTS</b>: Disk şifrələməsi üçün, tweak əsaslı, sektor səviyyəsində</li>
    <li>✅ <b>ChaCha20-Poly1305</b>: WireGuard, mobil cihazlar üçün optimallaşdırılmış</li>
</ul>
<p><i>AEAD rejimləri müasir kriptoqrafiyanın əsasını təşkil edir. Düzgün seçim və implementasiya təhlükəsizlik üçün kritik əhəmiyyət daşıyır.</i></p>
</div>

### 💭 Müzakirə sualları

1. Bu məşğələdə ən maraqlı tapdığınız nə oldu?
2. Hansı AEAD rejimi sizə daha məntiqli gəlir? Niyə?
3. Nonce təkrarı niyə GCM-də fəlakətdir, amma GCM-SIV-də niyə fəlakətli deyil? Hansı məlumat sızması hələ də mümkündür?
4. TLS 1.3-də niyə məhz GCM və ChaCha20-Poly1305 standartlaşdırılıb?
5. Disk şifrləməsində XTS rejiminin CBC-dən üstünlükləri nələrdir?
6. CCM rejimi niyə WPA2-də istifadə olunur, amma WPA3-də GCMP-yə keçilib?
7. AEAD rejimlərinin düzgün implementasiyası üçün ən vacib qaydalar hansılardır?


## 📚 Əlavə resurslar

* 📘 **NIST SP 800-38D (GCM):** [https://csrc.nist.gov/publications/detail/sp/800-38d/final](https://csrc.nist.gov/publications/detail/sp/800-38d/final)
* 📙 **NIST SP 800-38C (CCM):** [https://csrc.nist.gov/publications/detail/sp/800-38c/final](https://csrc.nist.gov/publications/detail/sp/800-38c/final)
* 📗 **NIST SP 800-38E (XTS):** [https://csrc.nist.gov/publications/detail/sp/800-38e/final](https://csrc.nist.gov/publications/detail/sp/800-38e/final)
* 📕 **RFC 8452 (GCM-SIV):** [https://tools.ietf.org/html/rfc8452](https://tools.ietf.org/html/rfc8452)
* 📘 **PyCryptodome sənədləşməsi:** [https://pycryptodome.readthedocs.io/](https://pycryptodome.readthedocs.io/)
* 📙 **Cryptography sənədləşməsi:** [https://cryptography.io/](https://cryptography.io/)
* 📗 **TLS 1.3 RFC 8446:** [https://tools.ietf.org/html/rfc8446](https://tools.ietf.org/html/rfc8446)
* 📘 **WireGuard White Paper:** [https://www.wireguard.com/papers/wireguard.pdf](https://www.wireguard.com/papers/wireguard.pdf)
* 📙 **IEEE 1619 (XTS):** [https://ieeexplore.ieee.org/document/8635988](https://ieeexplore.ieee.org/document/8635988)

---

<div style="background-color: #f0f0f0; padding: 20px; border-radius: 10px; text-align: center;">
<h2>✅ Məşğələ 12b tamamlandı!</h2>
<p>Bütün kodları və tapşırıq cavablarını növbəti məşğələyə qədər təqdim edin.</p>
<p><em>Kodlar aydın şərhlərlə yazılmalı və asan oxunmalıdır. Hər bir funksiyanın nə etdiyini izah edən şərhlər əlavə edin.</em></p>
<p style="font-size: 1.2em; margin-top: 15px;">🔐 <b>AEAD rejimləri - müasir kriptoqrafiyanın təməli!</b></p>
</div>